# 战国经济 BC：Colab T4
使用正常 Colab Notebook 运行。上传 ECS 合成的 150–250 回合示范归档；保留全部候选。每轮保存 best/last，结束后下载，运行中也可从文件面板取回。

In [ ]:
import subprocess, pathlib, os, torch
assert torch.cuda.is_available(), "请在运行时设置中选择 T4 GPU"
print(torch.__version__, torch.cuda.get_device_name(0))
ROOT = pathlib.Path('/content/zhanguo-economy-training')
if not ROOT.exists():
    subprocess.run(['git','clone','--depth','1','--branch','codex/rl-economy-bc','https://github.com/northwindlight/zhanguo.git',str(ROOT)], check=True)
os.chdir(ROOT)


In [ ]:
from google.colab import files
import tarfile, zipfile, shutil, json
uploaded = files.upload()  # 上传含 world-*.npz 的合成训练归档
unpacked = ROOT/'archive/colab-upload'
unpacked.mkdir(parents=True, exist_ok=True)
for name in uploaded:
    if tarfile.is_tarfile(name):
        with tarfile.open(name) as archive:
            archive.extractall(unpacked, filter='data')
    else:
        raise ValueError('请上传 .tar.gz 归档')
data = ROOT/'rl/runs/colab-t4/data'
data.mkdir(parents=True, exist_ok=True)
for source in unpacked.rglob('world-*.npz'):
    shutil.copy2(source, data/source.name)
from rl.economy.data import Dataset
from rl.economy.adapter import PROJECT_FINGERPRINT
dataset = Dataset(list(data.glob('world-*.npz')), fingerprint=PROJECT_FINGERPRINT)
assert all(m['turns'] >= 150 for m in dataset.metadata), '正式训练至少150回合'
valid = [s for s in (221,222) if (data/f'world-{s}.npz').exists()]
assert valid and len(dataset.metadata)>len(valid), '需要独立训练与验证世界'
print('worlds',len(dataset.metadata),'rows',len(dataset),'validation seeds',valid)


In [ ]:
import sys
out = ROOT/'rl/runs/colab-t4/model'
resume = ['--resume',str(out/'last.pt')] if (out/'last.pt').exists() else []
subprocess.run([sys.executable,'-m','rl.economy.train','--projects','--data',str(data),
    '--valid-seeds',*map(str,valid),'--device','cuda','--max-candidates','0',
    '--epochs','20','--batch','24','--threads','2','--out',str(out),*resume],check=True)


In [ ]:
result = ROOT/'archive/colab-t4-model'
result.parent.mkdir(parents=True, exist_ok=True)
shutil.make_archive(str(result),'zip',out)
files.download(str(result)+'.zip')
